In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, IntegerType,StringType, DoubleType)
from pyspark.sql.functions import (col, greatest, lit, when, sum, avg)

spark = (SparkSession.builder.appName("TelecomCustomerUsageAnalysis").getOrCreate())
spark

In [5]:
df = spark.read.csv('telecom_usage.csv',header=True,inferSchema=True)
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- plan_limit_gb: integer (nullable = true)



In [6]:
df.show()

+-----------+-------------+---------+------------+------------+-------------+
|customer_id|customer_name|     city|data_used_gb|call_minutes|plan_limit_gb|
+-----------+-------------+---------+------------+------------+-------------+
|       C101|         Arun|Bengaluru|          18|         220|           20|
|       C102|        Priya|  Chennai|          35|         310|           30|
|       C103|         Ravi|Hyderabad|          12|         150|           20|
|       C104|        Meena|Bengaluru|          42|         450|           30|
|       C105|        Kiran|   Mumbai|          28|         280|           25|
|       C106|        Divya|  Chennai|          15|         190|           20|
|       C107|       Sanjay|Hyderabad|          50|         500|           40|
|       C108|      Lakshmi|   Mumbai|          22|         240|           25|
|       C109|       Naveen|Bengaluru|          32|         350|           30|
|       C110|       Anitha|  Chennai|           8|         100| 

In [19]:
df = df.withColumn('excess_data_usage',greatest(col('data_used_gb')-col('plan_limit_gb'),lit(0.0)))

In [23]:
df = df.withColumn('excess_charge',col('excess_data_usage')*lit(10.0))

In [26]:
df = df.withColumn('usage_status',when(col('data_used_gb')>col('plan_limit_gb'),'Exceeded').otherwise('within limit'))

In [27]:
df.show()

+-----------+-------------+---------+------------+------------+-------------+-----------------+-------------+------------+
|customer_id|customer_name|     city|data_used_gb|call_minutes|plan_limit_gb|excess_data_usage|excess_charge|usage_status|
+-----------+-------------+---------+------------+------------+-------------+-----------------+-------------+------------+
|       C101|         Arun|Bengaluru|          18|         220|           20|              0.0|          0.0|within limit|
|       C102|        Priya|  Chennai|          35|         310|           30|              5.0|         50.0|    Exceeded|
|       C103|         Ravi|Hyderabad|          12|         150|           20|              0.0|          0.0|within limit|
|       C104|        Meena|Bengaluru|          42|         450|           30|             12.0|        120.0|    Exceeded|
|       C105|        Kiran|   Mumbai|          28|         280|           25|              3.0|         30.0|    Exceeded|
|       C106|   

In [29]:
city_data_usage = df.groupBy('city').agg(sum('data_used_gb'))
city_data_usage.show()

+---------+-----------------+
|     city|sum(data_used_gb)|
+---------+-----------------+
|  Chennai|               58|
|   Mumbai|               50|
|Bengaluru|               92|
|Hyderabad|               62|
+---------+-----------------+



In [43]:
top_five = df.select('customer_id','customer_name','data_used_gb').orderBy(col('data_used_gb').desc()).limit(5)
top_five.show()

+-----------+-------------+------------+
|customer_id|customer_name|data_used_gb|
+-----------+-------------+------------+
|       C107|       Sanjay|          50|
|       C104|        Meena|          42|
|       C102|        Priya|          35|
|       C109|       Naveen|          32|
|       C105|        Kiran|          28|
+-----------+-------------+------------+



In [42]:
city_summary = df.groupBy('city').agg(avg('data_used_gb'),sum('excess_charge'))
city_summary.show()

+---------+------------------+------------------+
|     city| avg(data_used_gb)|sum(excess_charge)|
+---------+------------------+------------------+
|  Chennai|19.333333333333332|              50.0|
|   Mumbai|              25.0|              30.0|
|Bengaluru|30.666666666666668|             140.0|
|Hyderabad|              31.0|             100.0|
+---------+------------------+------------------+



In [40]:
exceeded_customers = df.filter(col('data_used_gb')>col('plan_limit_gb'))
exceeded_customers.show()

+-----------+-------------+---------+------------+------------+-------------+-----------------+-------------+------------+
|customer_id|customer_name|     city|data_used_gb|call_minutes|plan_limit_gb|excess_data_usage|excess_charge|usage_status|
+-----------+-------------+---------+------------+------------+-------------+-----------------+-------------+------------+
|       C102|        Priya|  Chennai|          35|         310|           30|              5.0|         50.0|    Exceeded|
|       C104|        Meena|Bengaluru|          42|         450|           30|             12.0|        120.0|    Exceeded|
|       C105|        Kiran|   Mumbai|          28|         280|           25|              3.0|         30.0|    Exceeded|
|       C107|       Sanjay|Hyderabad|          50|         500|           40|             10.0|        100.0|    Exceeded|
|       C109|       Naveen|Bengaluru|          32|         350|           30|              2.0|         20.0|    Exceeded|
+-----------+---

In [44]:
df.write.mode('overwrite').parquet('output/processed_customers')

In [46]:
city_summary.write.mode('overwrite').parquet('output/city_summart')

In [47]:
exceeded_customers.write.mode('overwrite').parquet('output/exceeded_customers')

In [49]:
spark.read.parquet('output/processed_customers').show()

+-----------+-------------+---------+------------+------------+-------------+-----------------+-------------+------------+
|customer_id|customer_name|     city|data_used_gb|call_minutes|plan_limit_gb|excess_data_usage|excess_charge|usage_status|
+-----------+-------------+---------+------------+------------+-------------+-----------------+-------------+------------+
|       C101|         Arun|Bengaluru|          18|         220|           20|              0.0|          0.0|within limit|
|       C102|        Priya|  Chennai|          35|         310|           30|              5.0|         50.0|    Exceeded|
|       C103|         Ravi|Hyderabad|          12|         150|           20|              0.0|          0.0|within limit|
|       C104|        Meena|Bengaluru|          42|         450|           30|             12.0|        120.0|    Exceeded|
|       C105|        Kiran|   Mumbai|          28|         280|           25|              3.0|         30.0|    Exceeded|
|       C106|   